# PHY YOLO Pose Train

현재 수집 포맷의 port bbox와 4개 corner keypoint를 Ultralytics YOLO pose로 학습합니다.

- images: `images/<split>/<camera>/trial_<index>/*`
- annotations: `annotations/<split>/<camera>/trial_<index>/*.txt`
- metadata: `samples.jsonl`
- dataset config: `yolo_pose.yaml` (`kpt_shape: [4, 3]`)
- keypoint order: top-left, top-right, bottom-right, bottom-left

Class ID를 notebook에 고정하지 않고 `yolo_pose.yaml#names` 전체를 사용합니다. 따라서 SFP뿐 아니라 `sc_port` annotation이 추가되어도 같은 notebook으로 학습할 수 있습니다.

In [ ]:
import importlib.util
import json
import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").is_dir()
required_packages = {
    "hf_xet": "hf_xet",
    "huggingface_hub": "huggingface_hub",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "ultralytics": "ultralytics",
    "yaml": "PyYAML",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else :
    print("All required packages are already installed.")

import torch
import yaml


def find_src_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pixi.toml").is_file() and (path / "phy").is_dir():
            return path
        nested = path / "ws_aic" / "src"
        if (nested / "pixi.toml").is_file() and (nested / "phy").is_dir():
            return nested
    raise RuntimeError("ws_aic/src root를 찾지 못했습니다.")


if IS_KAGGLE:
    SRC_ROOT = Path("/kaggle/working")
    WS_ROOT = SRC_ROOT
else:
    SRC_ROOT = find_src_root(Path.cwd())
    WS_ROOT = SRC_ROOT.parent
    
DATASET_ROOT = WS_ROOT / "data" / "img2pos" / "phy_approach"
DATASET_NAMES = ("board-view", "near-port")
DATASET_DIRS = {name: DATASET_ROOT / name for name in DATASET_NAMES}

MODEL_NAME = "yolo11s-pose.pt"
MODEL_ROOT = WS_ROOT / "model" / "phy_yolo_pose"
RUN_NAME = "all_views"

EPOCHS = 200
IMGSZ = 640
BATCH = 16
WORKERS = 8
PATIENCE = 20
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

CUDA_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
TORCH_CUDA_ARCHES = set(torch.cuda.get_arch_list())
CUDA_ARCH_BY_DEVICE = {}
for index in range(CUDA_COUNT):
    major, minor = torch.cuda.get_device_capability(index)
    CUDA_ARCH_BY_DEVICE[index] = f"sm_{major}{minor}"
CUDA_DEVICES = [index for index, arch in CUDA_ARCH_BY_DEVICE.items() if arch in TORCH_CUDA_ARCHES]
TRAIN_DEVICE = CUDA_DEVICES[:2] if len(CUDA_DEVICES) >= 2 else (CUDA_DEVICES[0] if CUDA_DEVICES else "cpu")
if IS_KAGGLE and CUDA_COUNT >= 2:
    assert TRAIN_DEVICE == [0, 1], f"Kaggle T4 x2 is not supported by this PyTorch build: {CUDA_ARCH_BY_DEVICE}"
INFERENCE_DEVICE = TRAIN_DEVICE[0] if isinstance(TRAIN_DEVICE, list) else TRAIN_DEVICE
if isinstance(TRAIN_DEVICE, list) and BATCH % len(TRAIN_DEVICE):
    raise ValueError(f"BATCH={BATCH} must be divisible by GPU count={len(TRAIN_DEVICE)}")


CACHE_ROOT = WS_ROOT / ".cache"
os.environ.setdefault("HF_HOME", str(CACHE_ROOT / "huggingface"))
os.environ.setdefault("HF_HUB_CACHE", str(CACHE_ROOT / "huggingface" / "hub"))
os.environ.setdefault("YOLO_CONFIG_DIR", str(CACHE_ROOT / "ultralytics"))

RUNTIME = "kaggle" if IS_KAGGLE else "local"
print(f"runtime: {RUNTIME}")
print(f"SRC_ROOT: {SRC_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"DATASETS: {DATASET_DIRS}")
print(f"TRAIN_DEVICE: {TRAIN_DEVICE}")
for index in range(CUDA_COUNT):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}, architecture={CUDA_ARCH_BY_DEVICE[index]}, supported={index in CUDA_DEVICES}")

All required packages are already installed.
runtime: local
SRC_ROOT: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/src
DATASET_ROOT: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach
DATASETS: {'board-view': PosixPath('/home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach/board-view'), 'near-port': PosixPath('/home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach/near-port')}
TRAIN_DEVICE: cpu
GPU 0: NVIDIA GeForce GTX 1050, architecture=sm_61, supported=False


/home/swlinux/Desktop/workspace/CJ-Logistics-Challenge/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/swlinux/Desktop/workspace/CJ-Logistics-Challenge/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForce GTX 1050 with CUDA capability 

## Hugging Face dataset download

`team-physic/aic-align`의 `260812` revision 전체를 `DATASET_ROOT`에 병렬 다운로드합니다. `board-view`와 `near-port`를 모두 보존합니다. Private dataset이면 Local에서는 `HF_TOKEN` 또는 `hf auth login`, Kaggle에서는 Secrets의 `HF_TOKEN`을 자동으로 읽습니다.

In [ ]:
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

from huggingface_hub import snapshot_download

HF_DATASET_REPO = "team-physic/aic-align"
HF_DATASET_REVISION = "260812"
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_TOKEN_SOURCE = "environment" if HF_TOKEN else "anonymous"
if IS_KAGGLE and not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        HF_TOKEN_SOURCE = "Kaggle Secret"
    except Exception:
        HF_TOKEN = None

downloaded_dataset = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    revision=HF_DATASET_REVISION,
    local_dir=DATASET_ROOT,
    token=HF_TOKEN or None,
    max_workers=16,
)
print(f"Downloaded revision: {downloaded_dataset}")
print(f"HF authentication: {HF_TOKEN_SOURCE}")

## Combined dataset config

`board-view`와 `near-port`의 YAML schema를 검사합니다. 원본 파일은 수정하지 않고 두 dataset의 모든 split을 `model/phy_yolo_pose/combined_dataset` 아래 상대 symlink로 합칩니다.

In [ ]:
DATASET_CONFIGS = {}
CLASS_NAMES = None
KPT_SHAPE = None

for dataset_name, dataset_dir in DATASET_DIRS.items():
    yaml_path = dataset_dir / "yolo_pose.yaml"
    samples_path = dataset_dir / "samples.jsonl"
    annotations_dir = dataset_dir / "annotations"
    for required_path in (yaml_path, samples_path, annotations_dir):
        if not required_path.exists():
            raise FileNotFoundError(required_path)

    dataset_cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8")) or {}
    raw_names = dataset_cfg.get("names", {})
    class_names = (
        {index: str(name) for index, name in enumerate(raw_names)}
        if isinstance(raw_names, list)
        else {int(index): str(name) for index, name in raw_names.items()}
    )
    kpt_shape = tuple(map(int, dataset_cfg["kpt_shape"]))
    if kpt_shape != (4, 3):
        raise ValueError(f"{dataset_name}: expected kpt_shape [4, 3], got {list(kpt_shape)}")
    if not class_names:
        raise ValueError(f"{dataset_name}: yolo_pose.yaml#names is empty")
    if CLASS_NAMES is None:
        CLASS_NAMES, KPT_SHAPE = class_names, kpt_shape
    elif class_names != CLASS_NAMES or kpt_shape != KPT_SHAPE:
        raise ValueError(f"incompatible dataset schema: {dataset_name}")
    DATASET_CONFIGS[dataset_name] = {
        "dir": dataset_dir,
        "cfg": dataset_cfg,
        "samples": samples_path,
        "annotations": annotations_dir,
    }

KPT_COUNT, KPT_DIMS = KPT_SHAPE


def prepare_training_view() -> Path:
    view_dir = MODEL_ROOT / "combined_dataset"
    for dataset_name, dataset in DATASET_CONFIGS.items():
        for split in ("train", "val", "test"):
            if split not in dataset["cfg"]:
                continue
            image_target = dataset["dir"] / str(dataset["cfg"][split])
            label_target = dataset["annotations"] / split
            for kind, target in (("images", image_target), ("labels", label_target)):
                if not target.is_dir():
                    raise FileNotFoundError(target)
                link = view_dir / kind / split / dataset_name
                link.parent.mkdir(parents=True, exist_ok=True)
                relative_target = Path(os.path.relpath(target, link.parent))
                if link.is_symlink():
                    if link.readlink() != relative_target:
                        raise ValueError(f"unexpected symlink target: {link}")
                elif link.exists():
                    raise FileExistsError(link)
                else:
                    link.symlink_to(relative_target, target_is_directory=True)

    train_cfg = dict(next(iter(DATASET_CONFIGS.values()))["cfg"])
    train_cfg["path"] = str(view_dir)
    for split in ("train", "val", "test"):
        if any(split in dataset["cfg"] for dataset in DATASET_CONFIGS.values()):
            train_cfg[split] = f"images/{split}"
        else:
            train_cfg.pop(split, None)
    train_yaml = view_dir / "yolo_pose.yaml"
    train_yaml.write_text(
        yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    return train_yaml


print(f"datasets: {list(DATASET_CONFIGS)}")
print(f"classes: {CLASS_NAMES}")
print(f"kpt_shape: {[KPT_COUNT, KPT_DIMS]}")

## Combined dataset validation

두 dataset 각각의 split 경로, image/annotation 1:1 대응, YOLO pose row, class ID, 좌표, visibility, `samples.jsonl` 참조를 검사하고 합산 개수를 출력합니다.

In [ ]:
def split_dir(dataset_name: str, split: str) -> Path:
    dataset = DATASET_CONFIGS[dataset_name]
    value = Path(str(dataset["cfg"][split]))
    if value.is_absolute():
        raise ValueError(f"{dataset_name}/{split} path must be relative: {value}")
    return dataset["dir"] / value


def iter_images(path: Path) -> list[Path]:
    return sorted(
        candidate
        for candidate in path.rglob("*")
        if candidate.is_file() and candidate.suffix.lower() in IMAGE_EXTS
    )


def annotation_for(dataset_name: str, image_path: Path) -> Path:
    dataset = DATASET_CONFIGS[dataset_name]
    relative = image_path.relative_to(dataset["dir"] / "images")
    return (dataset["annotations"] / relative).with_suffix(".txt")


def validate_annotation(path: Path) -> list[str]:
    errors = []
    expected_fields = 5 + KPT_COUNT * KPT_DIMS
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != expected_fields:
            errors.append(f"{path}:{line_number}: expected {expected_fields} fields, got {len(fields)}")
            continue
        try:
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
        except ValueError:
            errors.append(f"{path}:{line_number}: invalid number")
            continue
        if class_id not in CLASS_NAMES:
            errors.append(f"{path}:{line_number}: unknown class_id {class_id}")
        bbox = values[:4]
        if any(value < 0.0 or value > 1.0 for value in bbox):
            errors.append(f"{path}:{line_number}: bbox outside [0, 1]")
        if bbox[2] <= 0.0 or bbox[3] <= 0.0:
            errors.append(f"{path}:{line_number}: bbox width/height must be positive")
        for index in range(4, len(values), KPT_DIMS):
            x, y, visibility = values[index:index + 3]
            if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
                errors.append(f"{path}:{line_number}: keypoint outside [0, 1]")
            if visibility not in (0.0, 1.0, 2.0):
                errors.append(f"{path}:{line_number}: visibility must be 0, 1, or 2")
    return errors


errors = []
dataset_images = set()
sample_images = set()
split_counts = Counter()
connector_counts = Counter()
camera_counts = Counter()

for dataset_name, dataset in DATASET_CONFIGS.items():
    for split in ("train", "val", "test"):
        if split not in dataset["cfg"]:
            continue
        image_root = split_dir(dataset_name, split)
        images = iter_images(image_root)
        split_counts[split] += len(images)
        dataset_images.update(
            f"{dataset_name}/{path.relative_to(dataset['dir']).as_posix()}" for path in images
        )
        if split in ("train", "val") and not images:
            errors.append(f"{dataset_name}/{split}: no images under {image_root}")
        expected_annotations = {annotation_for(dataset_name, path) for path in images}
        annotation_root = dataset["annotations"] / split
        actual_annotations = set(annotation_root.rglob("*.txt")) if annotation_root.exists() else set()
        errors.extend(f"missing annotation: {path}" for path in sorted(expected_annotations - actual_annotations))
        errors.extend(f"orphan annotation: {path}" for path in sorted(actual_annotations - expected_annotations))
        for annotation_path in sorted(actual_annotations):
            errors.extend(validate_annotation(annotation_path))

    for line_number, line in enumerate(dataset["samples"].read_text(encoding="utf-8").splitlines(), 1):
        try:
            row = json.loads(line)
        except json.JSONDecodeError as exc:
            errors.append(f"{dataset_name}/samples.jsonl:{line_number}: {exc}")
            continue
        connector_counts[str(row.get("connector", "unknown"))] += 1
        for camera, relative_image in row.get("images", {}).items():
            sample_images.add(f"{dataset_name}/{relative_image}")
            camera_counts[camera] += 1
            if not (dataset["dir"] / relative_image).is_file():
                errors.append(f"{dataset_name}/samples.jsonl:{line_number}: missing image {relative_image}")
            relative_annotation = row.get("annotations", {}).get(camera)
            if relative_annotation is None or not (dataset["dir"] / relative_annotation).is_file():
                errors.append(f"{dataset_name}/samples.jsonl:{line_number}: missing annotation for {camera}")

errors.extend(f"image missing from samples.jsonl: {path}" for path in sorted(dataset_images - sample_images))
errors.extend(f"samples.jsonl references unknown image: {path}" for path in sorted(sample_images - dataset_images))

print(f"split images: {dict(split_counts)}")
print(f"total images: {len(dataset_images)}")
print(f"connectors: {dict(connector_counts)}")
print(f"cameras: {dict(camera_counts)}")
print(f"classes: {CLASS_NAMES}")
if errors:
    preview = "\n".join(errors[:100])
    raise ValueError(f"Dataset validation failed with {len(errors)} errors.\n{preview}")
print(f"OK: images={len(dataset_images)}, expected label fields={5 + KPT_COUNT * KPT_DIMS}")

## Train all images

두 dataset의 같은 split끼리 결합합니다. `board-view/train + near-port/train`은 학습, 두 `val`은 검증, 두 `test`는 보류합니다. `classes` 인자를 넘기지 않아 YAML의 모든 class를 사용합니다. `TRAIN_DEVICE=[0, 1]`이면 Ultralytics DDP로 Kaggle T4 두 개를 사용합니다.

In [ ]:
from ultralytics import YOLO

TRAIN_YAML = prepare_training_view()
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(MODEL_ROOT),
    name=RUN_NAME,
    patience=PATIENCE,
    save=True,
    save_period=10,
    plots=True,
    device=TRAIN_DEVICE,
    workers=WORKERS,
    fliplr=0.0,
    flipud=0.0,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
print(f"run: {RUN_DIR}")
print(f"best: {BEST_PT}")

## Resume or validate

필요한 셀만 주석을 해제해 실행합니다.

In [ ]:
# 완전 재개
# model = YOLO(str(LAST_PT))
# train_results = model.train(resume=True)

# 학습 없이 validation
# model = YOLO(str(BEST_PT))
# metrics = model.val(
#     data=str(TRAIN_YAML), imgsz=IMGSZ, batch=BATCH, device=INFERENCE_DEVICE
# )
# print(metrics)

## Prediction preview

In [ ]:
import matplotlib.pyplot as plt

preview_images = []
for dataset_name in DATASET_NAMES:
    dataset_cfg = DATASET_CONFIGS[dataset_name]["cfg"]
    split = "val" if "val" in dataset_cfg else "train"
    preview_images.extend(iter_images(split_dir(dataset_name, split)))
sample = random.choice(preview_images)
model = YOLO(str(BEST_PT))
prediction = model.predict(
    source=str(sample),
    imgsz=IMGSZ,
    device=INFERENCE_DEVICE,
    save=True,
    project=str(MODEL_ROOT),
    name=f"{RUN_NAME}_predict",
)
annotated_bgr = prediction[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(annotated_bgr[..., ::-1])
plt.axis("off")
plt.title(sample.name)
print(sample)

## Upload best model and Model Card to Hugging Face

학습이 끝난 뒤 `BEST_PT`와 실행 당시 dataset·model·training config를 기록한 `README.md` Model Card를 model repository `team-physic/aic-approach`의 `0813-001` branch(revision)에 한 commit으로 업로드합니다. Branch가 이미 있으면 재사용하며 `main`은 변경하지 않습니다. Kaggle Secret 또는 환경 변수의 `HF_TOKEN`에는 해당 repository의 write 권한이 필요합니다. Local에서는 `hf auth login`으로 저장한 token도 사용할 수 있습니다.

In [ ]:
from huggingface_hub import CommitOperationAdd, HfApi

HF_MODEL_REPO = "team-physic/aic-approach"
HF_MODEL_REVISION = "0813-001"

if not BEST_PT.is_file():
    raise FileNotFoundError(f"trained checkpoint not found: {BEST_PT}")

split_rows = "\n".join(
    f"| `{split}` | {split_counts.get(split, 0)} |" for split in ("train", "val", "test")
)
class_rows = "\n".join(
    f"| {class_id} | `{class_name}` |" for class_id, class_name in sorted(CLASS_NAMES.items())
)
dataset_names = ", ".join(f"`{name}`" for name in DATASET_NAMES)
model_card = f"""---
library_name: ultralytics
tags:
- ultralytics
- yolo
- pose-estimation
datasets:
- {HF_DATASET_REPO}
---

# AIC Approach YOLO Pose ({HF_MODEL_REVISION})

## Artifact

- Checkpoint: `best.pt`
- Training run: `{RUN_NAME}`
- Runtime: `{RUNTIME}`

## Training dataset

- Repository: [`{HF_DATASET_REPO}`](https://huggingface.co/datasets/{HF_DATASET_REPO})
- Revision: `{HF_DATASET_REVISION}`
- Included directories: {dataset_names}
- Total images: {len(dataset_images)}
- Connector records: `{dict(connector_counts)}`
- Camera records: `{dict(camera_counts)}`

| Split | Images |
|---|---:|
{split_rows}

## Model

- Framework: Ultralytics YOLO Pose
- Initial checkpoint: `{MODEL_NAME}`
- Output: bounding box + {KPT_COUNT} keypoints × {KPT_DIMS} values

| Class ID | Class name |
|---:|---|
{class_rows}

## Training configuration

| Parameter | Value |
|---|---|
| `epochs` | `{EPOCHS}` |
| `imgsz` | `{IMGSZ}` |
| `batch` | `{BATCH}` |
| `workers` | `{WORKERS}` |
| `patience` | `{PATIENCE}` |
| `device` | `{TRAIN_DEVICE}` |
| `save_period` | `10` |
| `fliplr` | `0.0` |
| `flipud` | `0.0` |

api = HfApi(token=HF_TOKEN or None)
identity = api.whoami()
api.create_branch(
    repo_id=HF_MODEL_REPO,
    branch=HF_MODEL_REVISION,
    repo_type="model",
    exist_ok=True,
)
commit = api.create_commit(
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    revision=HF_MODEL_REVISION,
    operations=[
        CommitOperationAdd(path_in_repo="best.pt", path_or_fileobj=BEST_PT),
        CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=model_card.encode("utf-8")),
    ],
    commit_message=f"Upload {RUN_NAME} checkpoint and model card",
)
print(f"HF account: {identity['name']}")
print(f"Uploaded checkpoint and Model Card: {commit.commit_url}")
print(f"Model: https://huggingface.co/{HF_MODEL_REPO}/tree/{HF_MODEL_REVISION}")